In [1]:
pip install groq pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["GROQ_API_KEY"] = "enter_api_key"

In [3]:
import os
import json
import time
import pandas as pd
from groq import Groq

In [4]:
# ── 1. CONFIG ────────────────────────────────────────────────

# Read the API key from the environment variable
API_KEY = os.environ.get("GROQ_API_KEY")

# Use Llama 3.1 8B for fast, high-limit inference
MODEL_NAME = "llama-3.1-8b-instant" 

# Path to the dataset downloaded from GitHub
INPUT_FILE = "dataset_clean.csv"

# Name of the output CSV file you will submit to GitHub
OUTPUT_FILE = "output_model1.csv"

# Groq is very fast; 1 second is enough to avoid rate limits
SLEEP_BETWEEN_BATCHES = 1

In [5]:
# ── 2. VALIDATE API KEY ──────────────────────────────────────

if not API_KEY:
    raise ValueError(
        "No API key found!\n"
        "Please run this in a cell above:\n"
        "  os.environ['GROQ_API_KEY'] = 'gsk_...' "
    )

In [6]:
# ── 3. SETUP GROQ ──────────────────────────────────────────

# Initialize the Groq client
client = Groq(api_key=API_KEY)

print(f"Model loaded: {MODEL_NAME}")

Model loaded: llama-3.1-8b-instant


In [7]:
# ── 4. LOAD DATASET ──────────────────────────────────────────

# Read the CSV into a DataFrame
df = pd.read_csv(INPUT_FILE)

# Show column names and a preview
print(f"\nDataset loaded: {len(df)} rows")
print("Column names:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3))


Dataset loaded: 643 rows
Column names: ['id', 'prompt']

First 3 rows:
             id                                             prompt
0  CORP-TAX-001  Was ist die steuerliche Bemessungsgrundlage fü...
1  CORP-TAX-002  Welche steuerlichen Konsequenzen hat es, wenn ...
2  CORP-TAX-003  Welche Körperschaften sind verpflichtet, sämtl...


In [8]:
# ── 5. SYSTEM INSTRUCTIONS ────────────────────────────

SYSTEM_INSTRUCTION = (
    "You are an expert on Austrian tax law. "
    "Answer the following questions accurately and concisely. "
    "Limit each answer to approximately 150 words. "
    "Answer in the same language the question is asked in. "
    "IMPORTANT: You must return your output ONLY as a JSON object containing a list called 'results', "
    "where each item has an 'id' and an 'answer'."
)

In [9]:
# ── 6. BATCHED LOOP THROUGH QUESTIONS ────────────────────────
QUESTION_COLUMN = "prompt"  # Matches your CSV column name
BATCH_SIZE = 10 
results = []

print(f"\nStarting batch inference on {len(df)} questions...")

for i in range(0, len(df), BATCH_SIZE):
    chunk = df.iloc[i : i + BATCH_SIZE]
    
    # Build a single combined prompt for the batch
    combined_prompt = "Process the following questions and return the answers in the requested JSON format:\n\n"
    for _, row in chunk.iterrows():
        combined_prompt += f"ID: {row['id']} | Question: {row[QUESTION_COLUMN]}\n"

    print(f"[{min(i + BATCH_SIZE, len(df))}/{len(df)}] Processing batch...")

    success = False
    retries = 0
    
    while not success and retries < 3:
        try:
            # Groq API Call
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": combined_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0,
            )
            
            # Parse the JSON response
            raw_content = response.choices[0].message.content
            batch_data = json.loads(raw_content)
            
            # Extract the list from the JSON object
            if "results" in batch_data:
                results.extend(batch_data["results"])
            else:
                # Fallback if AI didn't use the 'results' key
                results.extend(list(batch_data.values())[0])
            
            success = True
            
        except Exception as e:
            print(f"  -> Error: {e}. Retrying in 10s...")
            time.sleep(10)
            retries += 1

    # Brief pause between batches
    time.sleep(SLEEP_BETWEEN_BATCHES)


Starting batch inference on 643 questions...
[10/643] Processing batch...
[20/643] Processing batch...
[30/643] Processing batch...
[40/643] Processing batch...
[50/643] Processing batch...
  -> Error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01knffep6ee4ytakh2ep9e2ykz` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4362, Requested 1640. Please try again in 20ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Retrying in 10s...
[60/643] Processing batch...
[70/643] Processing batch...
[80/643] Processing batch...
[90/643] Processing batch...
[100/643] Processing batch...
[110/643] Processing batch...
[120/643] Processing batch...
[130/643] Processing batch...
[140/643] Processing batch...
[150/643] Processing batch...
[160/643] Processing batch...
[170/643] Processing batch...
[180/643] Processing

In [10]:
# ── 7. SAVE OUTPUT ───────────────────────────────────────────

# Convert results to a DataFrame
output_df = pd.DataFrame(results)

# Ensure the output matches the required assignment format
output_df = output_df[["id", "answer"]] 
output_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nDone! Saved {len(output_df)} answers to '{OUTPUT_FILE}'")
print("\nOutput preview:")
print(output_df.head(3))


Done! Saved 643 answers to 'submission_step1_groq.csv'

Output preview:
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Wenn eine Körperschaft ein zinsloses Darlehen ...
2  CORP-TAX-003  Körperschaften, die gemäß § 1 Abs. 1 Z 1 KStG ...
